# 02 — Combined RQ2: uncertainty reliability and calibration

**RQ2:** How well do VLM confidence and uncertainty estimates reflect the reliability of robot-safety classification predictions?

This final notebook analyzes the combined **HuRoN + PAL + CrowdBot** clean freeze:

- 25 bags and 34,912 physical frames per model/approach/history configuration
- 3 models: Qwen, InternVL, and LLaVA
- 2 approaches: AppFr and AppOd
- 16 histories: H02 through H32
- 96 combined clean files

The notebook contains only the three inferential analyses needed for RQ2:

1. **RQ2-A — Expected reliability pattern:** within every model, approach, history, and uncertainty score, determine whether the bag-level correctness correlation follows the expected direction.
2. **RQ2-B — Model comparison by setting:** within every approach and history, compare the three models on MSP, PCS, Deep Gini, entropy, ECE, and MCE.
3. **RQ2-C — Exploratory global model comparison:** compare the models across all matched bag × approach × history blocks for each of the six metrics.

All inferential tests use bag-level values. Pooled 34,912-frame correlations in RQ2-A are descriptive only. Every result table is written to CSV and LaTeX.

In [ ]:
from pathlib import Path
from itertools import combinations
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import friedmanchisquare, pointbiserialr, rankdata, wilcoxon

HERE = Path.cwd().resolve()
ROOT = next((
    p for p in [HERE, *HERE.parents]
    if (p / "case_studies").is_dir() and (p / "analysis").is_dir()
), None)

if ROOT is None:
    raise RuntimeError("Could not locate repository root.")

CLEAN = ROOT / "analysis" / "combined" / "clean"
OUT = ROOT / "results" / "rq2"
TEX = OUT / "latex"
OUT.mkdir(parents=True, exist_ok=True)
TEX.mkdir(parents=True, exist_ok=True)

MODELS = ["qwen", "internvl", "llava"]
APPROACHES = ["appfr", "appod"]
HISTORIES = list(range(2, 33, 2))
HISTORY_LABELS = [f"H{h:02d}" for h in HISTORIES]
LABELS = ["safe", "potentially_unsafe", "unsafe"]
PROBS = ["prob_safe", "prob_potentially_unsafe", "prob_unsafe"]
SCORES = ["msp", "pcs", "deep_gini", "entropy"]
METRICS = SCORES + ["ece", "mce"]

EXPECTED_ROWS, EXPECTED_BAGS = 34_912, 25
EXPECTED_CASE_BAGS = {"huron": 15, "pal": 3, "crowdbot": 7}
EXPECTED_CASE_ROWS = {"huron": 16_862, "pal": 9_249, "crowdbot": 8_801}
N_BINS, ALPHA, MIN_COMPLETE_BAGS = 10, 0.05, 5

MODEL_NAMES = {"qwen": "Qwen", "internvl": "InternVL", "llava": "LLaVA"}
APPROACH_NAMES = {"appfr": "AppFr", "appod": "AppOd"}
METRIC_NAMES = {
    "msp": "MSP r", "pcs": "PCS r", "deep_gini": "Deep Gini r",
    "entropy": "Entropy r", "ece": "ECE", "mce": "MCE",
}
MODEL_CODES = {"qwen": "Q", "internvl": "I", "llava": "L"}

# Larger oriented values always indicate better uncertainty reliability/calibration.
ORIENTATION = {
    "msp": 1.0, "pcs": 1.0, "deep_gini": -1.0,
    "entropy": -1.0, "ece": -1.0, "mce": -1.0,
}
DIRECTION = {
    "msp": "higher positive correctness correlation is better",
    "pcs": "higher positive correctness correlation is better",
    "deep_gini": "more negative correctness correlation is better",
    "entropy": "more negative correctness correlation is better",
    "ece": "lower calibration error is better",
    "mce": "lower calibration error is better",
}

EXPORTED = []

def clean_file(model, approach, history):
    candidates = [
        CLEAN / model / f"{model}_{approach}_h{history:02d}.csv",
        CLEAN / model / approach / f"H{history:02d}.csv",
    ]
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError("Missing combined clean file; tried: " + ", ".join(map(str, candidates)))

def export(df, name, caption, index=False):
    df.to_csv(OUT / f"{name}.csv", index=index)
    kwargs = dict(
        index=index, escape=True, na_rep="--", float_format=lambda x: f"{x:.6g}",
        caption=caption, label=f"tab:{name.replace('_', '-')}",
        multicolumn=True, multicolumn_format="c",
    )
    kwargs["longtable" if len(df) > 80 else "position"] = True if len(df) > 80 else "tbp"
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", FutureWarning)
        latex = df.to_latex(**kwargs)
    (TEX / f"{name}.tex").write_text(latex, encoding="utf-8")
    if name not in EXPORTED:
        EXPORTED.append(name)

def normalize_label(series):
    return (series.astype(str).str.strip().str.lower()
            .str.replace("-", "_", regex=False).str.replace(" ", "_", regex=False))

def add_scores(df):
    df = df.copy()
    p = df[PROBS].to_numpy(float)
    ordered = np.sort(p, axis=1)
    df["correct_binary"] = (df["ground_truth"] == df["predicted_label"]).astype(int)
    df["msp"] = p.max(axis=1)
    df["pcs"] = ordered[:, -1] - ordered[:, -2]
    df["deep_gini"] = 1.0 - np.square(p).sum(axis=1)
    df["entropy"] = -(p * np.log(p + 1e-12)).sum(axis=1)
    return df

def safe_point_biserial(correct, values):
    correct, values = np.asarray(correct), np.asarray(values, float)
    valid = pd.notna(correct) & np.isfinite(values)
    correct, values = correct[valid], values[valid]
    if len(correct) < 3 or np.unique(correct).size < 2 or np.unique(values).size < 2:
        return np.nan, np.nan
    if np.linalg.norm(values - values.mean()) < 1e-13 * max(abs(values.mean()), 1.0):
        return np.nan, np.nan
    result = pointbiserialr(correct, values)
    return float(result.statistic), float(result.pvalue)

def calibration_errors(confidence, correct, n_bins=N_BINS):
    confidence, correct = np.asarray(confidence, float), np.asarray(correct, float)
    ids = np.minimum((confidence * n_bins).astype(int), n_bins - 1)
    gaps, counts = [], []
    for bin_id in range(n_bins):
        mask = ids == bin_id
        if mask.any():
            gaps.append(abs(correct[mask].mean() - confidence[mask].mean()))
            counts.append(mask.sum())
    gaps, counts = np.asarray(gaps), np.asarray(counts)
    return float(np.sum(gaps * counts) / len(correct)), float(gaps.max())

def holm_adjust(p_values):
    p = np.asarray(p_values, float)
    adjusted = np.full(len(p), np.nan)
    valid = np.flatnonzero(np.isfinite(p))
    order = valid[np.argsort(p[valid])]
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (len(order) - rank) * p[index])
        adjusted[index] = min(running, 1.0)
    return adjusted

def rank_biserial(x, y):
    d = np.asarray(x, float) - np.asarray(y, float)
    d = d[np.isfinite(d) & (d != 0)]
    if not len(d):
        return 0.0
    ranks = rankdata(np.abs(d), method="average")
    return float((ranks[d > 0].sum() - ranks[d < 0].sum()) / ranks.sum())

def safe_wilcoxon(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    if np.allclose(x, y):
        return 0.0, 1.0
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided", method="auto")
    return float(result.statistic), float(result.pvalue)

def safe_friedman(wide):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            result = friedmanchisquare(*[wide[m] for m in MODELS])
        statistic, p_value = float(result.statistic), float(result.pvalue)
        return (statistic, p_value) if np.isfinite(statistic + p_value) else (0.0, 1.0)
    except ValueError:
        return 0.0, 1.0

def model_pairwise_tests(wide, metadata):
    rows = []
    for model_1, model_2 in combinations(MODELS, 2):
        x, y = wide[model_1], wide[model_2]
        statistic, raw_p = safe_wilcoxon(x, y)
        effect = rank_biserial(x, y)
        rows.append({
            **metadata, "model_1": model_1, "model_2": model_2, "n_pairs": len(wide),
            "median_oriented_difference_model1_minus_model2": (x - y).median(),
            "wilcoxon_statistic": statistic, "raw_p": raw_p,
            "rank_biserial_model1_minus_model2": effect,
            "descriptive_better_model": model_1 if effect > 0 else model_2 if effect < 0 else "tie",
        })
    for row, p_holm in zip(rows, holm_adjust([row["raw_p"] for row in rows])):
        row["holm_p"] = p_holm
        row["significant_holm"] = bool(p_holm < ALPHA)
        row["significant_better_model"] = (
            row["descriptive_better_model"] if p_holm < ALPHA else "No difference"
        )
    return rows

def strict_winner(comparisons):
    winners = []
    for model in MODELS:
        involving = comparisons[(comparisons["model_1"] == model) | (comparisons["model_2"] == model)]
        if len(involving) == 2 and involving["significant_better_model"].eq(model).all():
            winners.append(model)
    return winners[0] if len(winners) == 1 else "No unique winner"

print("Input:", CLEAN)
print("Output:", OUT)
print("Design: 3 models × 2 approaches × 16 histories = 96 clean files")


## Prepare the analysis data

This cell validates the 96-file combined clean freeze, calculates MSP, PCS, Deep Gini, entropy, ECE, and MCE, and constructs the 25-bag matched analysis table. The saved predicted label remains the classification decision; probability argmax is not substituted.

In [ ]:
REQUIRED = {"case_study", "global_bag_id", "frame_id", "ground_truth", "predicted_label", *PROBS}
validation_rows, correlation_rows, bag_metric_rows = [], [], []
reference_identity = reference_bags = None

for model in MODELS:
    for approach in APPROACHES:
        for history in HISTORIES:
            path = clean_file(model, approach, history)
            df = pd.read_csv(path, low_memory=False)
            name = f"{model} {approach} H{history:02d}"
            assert REQUIRED.issubset(df), f"{name}: missing {sorted(REQUIRED - set(df))}"

            for column in ["case_study", "ground_truth", "predicted_label"]:
                df[column] = normalize_label(df[column])
            df["global_bag_id"] = df["global_bag_id"].astype(str)

            assert len(df) == EXPECTED_ROWS, f"{name}: wrong row count"
            assert df["global_bag_id"].nunique() == EXPECTED_BAGS, f"{name}: wrong bag count"
            assert not df[["global_bag_id", "frame_id"]].isna().any().any(), f"{name}: missing identity"
            assert not df.duplicated(["global_bag_id", "frame_id"]).any(), f"{name}: duplicate frames"
            assert set(df["ground_truth"]) <= set(LABELS), f"{name}: invalid ground truth"
            assert set(df["predicted_label"]) <= set(LABELS), f"{name}: invalid prediction"

            case_bags = df.groupby("case_study")["global_bag_id"].nunique().astype(int).to_dict()
            case_rows = df.groupby("case_study").size().astype(int).to_dict()
            assert case_bags == EXPECTED_CASE_BAGS, f"{name}: case-study bags {case_bags}"
            assert case_rows == EXPECTED_CASE_ROWS, f"{name}: case-study rows {case_rows}"

            p = df[PROBS].to_numpy(float)
            assert np.isfinite(p).all() and ((0 <= p) & (p <= 1)).all(), f"{name}: invalid probability"
            assert np.isclose(p.sum(axis=1), 1.0, atol=1e-5).all(), f"{name}: probability sums"
            if "prediction_valid" in df:
                valid = df["prediction_valid"]
                if valid.dtype != bool:
                    valid = valid.astype(str).str.strip().str.lower().isin(["true", "1", "1.0"])
                assert valid.all(), f"{name}: invalid prediction rows"
            if "repair_status" in df:
                assert not df["repair_status"].astype(str).eq("provisional_one_hot").any(), name

            identity = (df[["global_bag_id", "frame_id", "ground_truth"]].astype(str)
                        .sort_values(["global_bag_id", "frame_id"]).reset_index(drop=True))
            bags = set(df["global_bag_id"])
            if reference_identity is None:
                reference_identity, reference_bags = identity, bags
            else:
                assert bags == reference_bags and identity.equals(reference_identity), f"{name}: identity mismatch"

            df = add_scores(df)
            key = {"model": model, "approach": approach,
                   "history": f"H{history:02d}", "history_frames": history}
            for score in SCORES:
                r_pb, p_value = safe_point_biserial(df["correct_binary"], df[score])
                correlation_rows.append({**key, "score": score, "r_pb": r_pb,
                                         "p_value": p_value, "direction": DIRECTION[score]})

            for (case_study, bag_id), bag in df.groupby(["case_study", "global_bag_id"], sort=True):
                bag_key = {**key, "case_study": case_study, "global_bag_id": bag_id,
                           "frames": len(bag), "accuracy": bag["correct_binary"].mean()}
                for score in SCORES:
                    r_pb, p_value = safe_point_biserial(bag["correct_binary"], bag[score])
                    bag_metric_rows.append({
                        **bag_key, "metric": score, "original_value": r_pb,
                        "oriented_value": r_pb * ORIENTATION[score] if np.isfinite(r_pb) else np.nan,
                        "correlation_p_value": p_value, "direction": DIRECTION[score],
                        "valid_value": bool(np.isfinite(r_pb)),
                    })
                ece, mce = calibration_errors(bag["msp"], bag["correct_binary"])
                for metric, value in [("ece", ece), ("mce", mce)]:
                    bag_metric_rows.append({
                        **bag_key, "metric": metric, "original_value": value,
                        "oriented_value": ORIENTATION[metric] * value,
                        "correlation_p_value": np.nan, "direction": DIRECTION[metric],
                        "valid_value": True,
                    })

            validation_rows.append({**key, "input_file": str(path.relative_to(ROOT)),
                                    "rows": len(df), "bags": df["global_bag_id"].nunique()})

validation = pd.DataFrame(validation_rows)
correlations = pd.DataFrame(correlation_rows)
bag_metrics = pd.DataFrame(bag_metric_rows)

assert len(validation) == 96 and validation["rows"].eq(EXPECTED_ROWS).all()
assert validation["bags"].eq(EXPECTED_BAGS).all()
assert len(correlations) == 96 * len(SCORES) == 384
assert len(bag_metrics) == 96 * EXPECTED_BAGS * len(METRICS) == 14_400

display(validation.groupby(["model", "approach"]).agg(
    clean_files=("input_file", "size"), rows_per_file=("rows", "first"),
    bags_per_file=("bags", "first")))
print(f"Ready: {len(validation)} files; {len(bag_metrics):,} bag–metric rows.")

## RQ2-A — Do uncertainty estimates follow the expected reliability pattern?

For each model × approach × uncertainty score × history, this analysis tests the 25 bag-level point-biserial correlations against zero using a two-sided one-sample Wilcoxon test. Signs are oriented so positive always means the expected pattern: MSP/PCS should correlate positively with correctness, while Deep Gini/entropy should correlate negatively. Holm correction is applied across the 16 histories within each model × approach × score family.

The two wide tables report the descriptive pooled 34,912-frame correlation followed by the bag-level decision: **E** = significant expected pattern, **O** = significant opposite pattern, **NS** = not significant, and **NA** = too few valid bags. RQ2-A does not compare AppFr against AppOd.

In [ ]:
RQ2A_NAMES = {"msp": "MSP", "pcs": "PCS", "deep_gini": "DG", "entropy": "ET"}
DECISION_CODES = {"Expected": "E", "Opposite": "O", "No clear pattern": "NS",
                  "Insufficient bags": "NA"}
pooled_lookup = correlations.set_index(["model", "approach", "history_frames", "score"])["r_pb"]

rows = []
for model in MODELS:
    for approach in APPROACHES:
        for score in SCORES:
            for history in HISTORIES:
                part = bag_metrics.query(
                    "model == @model and approach == @approach and metric == @score "
                    "and history_frames == @history"
                )
                values = part["oriented_value"].dropna().to_numpy(float)
                performed = len(values) >= MIN_COMPLETE_BAGS
                statistic, raw_p = (safe_wilcoxon(values, np.zeros(len(values)))
                                    if performed else (np.nan, np.nan))
                rows.append({
                    "model": model, "approach": approach, "score": score,
                    "expected_direction": DIRECTION[score], "history": f"H{history:02d}",
                    "history_frames": history,
                    "pooled_r_34912_frames": pooled_lookup.loc[model, approach, history, score],
                    "n_valid_bags": len(values), "expected_sign_bags": int((values > 0).sum()),
                    "opposite_sign_bags": int((values < 0).sum()),
                    "median_bag_r_original": part["original_value"].median(),
                    "median_bag_r_oriented": np.median(values) if len(values) else np.nan,
                    "wilcoxon_statistic": statistic, "raw_p": raw_p,
                    "rank_biserial_vs_zero": (rank_biserial(values, np.zeros(len(values)))
                                               if len(values) else np.nan),
                    "test_performed": performed,
                })

rq2a = pd.DataFrame(rows)
rq2a["holm_p"] = rq2a.groupby(["model", "approach", "score"])["raw_p"].transform(holm_adjust)
rq2a["significant_holm"] = rq2a["holm_p"] < ALPHA
rq2a["decision"] = "No clear pattern"
rq2a.loc[~rq2a["test_performed"], "decision"] = "Insufficient bags"
rq2a.loc[rq2a["significant_holm"] & rq2a["rank_biserial_vs_zero"].gt(0), "decision"] = "Expected"
rq2a.loc[rq2a["significant_holm"] & rq2a["rank_biserial_vs_zero"].lt(0), "decision"] = "Opposite"
rq2a["pooled_sign_expected"] = rq2a["pooled_r_34912_frames"] * rq2a["score"].map(ORIENTATION) > 0
rq2a["table_value"] = [
    "NA" if not np.isfinite(r) else f"{r:.3f} [{DECISION_CODES[d]}]"
    for r, d in zip(rq2a["pooled_r_34912_frames"], rq2a["decision"])
]

rq2a_tables = {}
for approach in APPROACHES:
    table = (rq2a.query("approach == @approach")
             .pivot(index="history", columns=["model", "score"], values="table_value")
             .reindex(index=HISTORY_LABELS,
                      columns=pd.MultiIndex.from_product([MODELS, SCORES])))
    table.columns = pd.MultiIndex.from_tuples(
        [(MODEL_NAMES[m], RQ2A_NAMES[s]) for m, s in table.columns],
        names=["Model", "Uncertainty score"],
    )
    table.index.name = "History"
    rq2a_tables[approach] = table
    export(table, f"rq2a_expected_pattern_{approach}",
           f"Pooled r and bag-level expected-pattern decision for {APPROACH_NAMES[approach]}.",
           index=True)

rq2a_summary = (rq2a.groupby(["model", "approach", "score", "decision"])
                .size().reset_index(name="number_of_histories"))
rq2a_overall = (rq2a.groupby(["model", "approach", "decision"])
                .size().unstack(fill_value=0).reset_index())
export(rq2a.drop(columns="table_value"), "rq2a_expected_pattern_tests",
       "RQ2-A bag-level expected-direction tests.")
export(rq2a_summary, "rq2a_expected_pattern_summary_by_score",
       "RQ2-A expected-pattern decisions by score.")
export(rq2a_overall, "rq2a_expected_pattern_summary_overall",
       "RQ2-A overall expected-pattern decisions.")

for approach in APPROACHES:
    print(f"\nCorrect binary vs numeric score — {APPROACH_NAMES[approach]}")
    display(rq2a_tables[approach])
display(rq2a_summary)
print("RQ2-A tests:", len(rq2a))

## RQ2-B — Which model performs best within each approach and history?

For each of the 6 metrics × 2 approaches × 16 histories, a Friedman test compares Qwen, InternVL, and LLaVA using matched bags: **192 omnibus tests**. Correlations are oriented toward their expected sign; ECE and MCE are negated so larger oriented values always mean better performance. Holm correction is applied across histories within each approach × metric family.

Only a significant Holm-adjusted Friedman result triggers the three paired Wilcoxon post-hoc comparisons, which receive a second Holm correction within that three-comparison family. A model is shown as the unique winner only if it significantly beats both alternatives.

In [ ]:
rq2b_cache, omnibus_rows = {}, []
for metric in METRICS:
    for approach in APPROACHES:
        for history in HISTORIES:
            part = bag_metrics.query(
                "metric == @metric and approach == @approach and history_frames == @history"
            )
            original = part.pivot(index="global_bag_id", columns="model", values="original_value").reindex(columns=MODELS)
            oriented = part.pivot(index="global_bag_id", columns="model", values="oriented_value").reindex(columns=MODELS).dropna()
            original = original.reindex(index=oriented.index)
            n, performed = len(oriented), len(oriented) >= MIN_COMPLETE_BAGS
            statistic, raw_p = safe_friedman(oriented) if performed else (np.nan, np.nan)
            rq2b_cache[(metric, approach, history)] = oriented
            omnibus_rows.append({
                "metric": metric, "direction": DIRECTION[metric], "approach": approach,
                "history": f"H{history:02d}", "history_frames": history,
                "n_complete_bags": n, "test_performed": performed,
                **{f"{m}_median_original": original[m].median() if n else np.nan for m in MODELS},
                **{f"{m}_median_oriented": oriented[m].median() if n else np.nan for m in MODELS},
                "best_median_model": (max(MODELS, key=lambda m: oriented[m].median())
                                      if n else "unavailable"),
                "friedman_chi2": statistic, "degrees_of_freedom": len(MODELS) - 1,
                "raw_p": raw_p,
                "kendalls_w": statistic / (n * (len(MODELS) - 1)) if performed else np.nan,
            })

rq2b_friedman = pd.DataFrame(omnibus_rows)
rq2b_friedman["holm_p"] = rq2b_friedman.groupby(["approach", "metric"])["raw_p"].transform(holm_adjust)
rq2b_friedman["significant_holm"] = rq2b_friedman["holm_p"] < ALPHA

posthoc_rows = []
for row in rq2b_friedman.itertuples(index=False):
    if row.significant_holm:
        posthoc_rows.extend(model_pairwise_tests(
            rq2b_cache[(row.metric, row.approach, row.history_frames)],
            {"metric": row.metric, "direction": row.direction, "approach": row.approach,
             "history": row.history, "history_frames": row.history_frames,
             "friedman_holm_p": row.holm_p},
        ))
rq2b_posthoc = pd.DataFrame(posthoc_rows)

winner_rows = []
for row in rq2b_friedman.itertuples(index=False):
    winner, reason = "No unique winner", "Omnibus not significant"
    if row.significant_holm:
        comparisons = rq2b_posthoc.query(
            "metric == @row.metric and approach == @row.approach and history == @row.history"
        )
        winner = strict_winner(comparisons)
        reason = ("Significantly better than both alternatives" if winner in MODELS
                  else "No model significantly beat both alternatives")
    winner_rows.append({
        "metric": row.metric, "approach": row.approach, "history": row.history,
        "history_frames": row.history_frames, "unique_winner": winner,
        "winner_code": MODEL_CODES.get(winner, "—"), "decision_reason": reason,
    })
rq2b_winners = pd.DataFrame(winner_rows)

row_order = pd.MultiIndex.from_product(
    [[APPROACH_NAMES[a] for a in APPROACHES], [METRIC_NAMES[m] for m in METRICS]],
    names=["Approach", "Metric"],
)
rq2b_winner_matrix = (rq2b_winners.assign(
    Approach=rq2b_winners["approach"].map(APPROACH_NAMES),
    Metric=rq2b_winners["metric"].map(METRIC_NAMES),
).pivot(index=["Approach", "Metric"], columns="history", values="winner_code")
  .reindex(index=row_order, columns=HISTORY_LABELS))
rq2b_winner_summary = (rq2b_winners.groupby(["metric", "approach", "unique_winner"])
                       .size().reset_index(name="number_of_histories"))

export(rq2b_friedman, "rq2b_model_friedman",
       "RQ2-B model comparisons by uncertainty/calibration metric, approach, and history.")
export(rq2b_posthoc, "rq2b_model_pairwise_wilcoxon",
       "RQ2-B gated pairwise Wilcoxon-Holm model comparisons.")
export(rq2b_winners, "rq2b_model_winners_by_history",
       "RQ2-B strict unique-winner decisions by setting.")
export(rq2b_winner_matrix, "rq2b_significant_model_winners",
       "RQ2-B winners: Q=Qwen, I=InternVL, L=LLaVA, dash=no unique winner.", index=True)
export(rq2b_winner_summary, "rq2b_model_winner_summary",
       "Counts of RQ2-B unique model winners across histories.")

assert len(rq2b_friedman) == 192
assert len(rq2b_posthoc) == 3 * int(rq2b_friedman["significant_holm"].sum())
display(rq2b_winner_matrix)
display(rq2b_winner_summary)
print("Friedman tests:", len(rq2b_friedman))
print("Significant Friedman tests:", int(rq2b_friedman["significant_holm"].sum()))
print("Gated pairwise Wilcoxon tests:", len(rq2b_posthoc))

## RQ2-C — Exploratory global model comparison

For each metric, the unit is a matched **case-study bag × approach × history** block. There are 25 × 2 × 16 = 800 possible blocks per metric. A block is included only when all three models have a valid value; this complete-case rule can exclude correlation blocks when a bag has constant correctness or a near-constant uncertainty score. ECE and MCE remain defined for all bags.

One Friedman test is performed for each of the six metrics and Holm-corrected across those six tests. Significant omnibus results trigger three paired Wilcoxon tests with within-metric Holm correction. This analysis is explicitly exploratory because the same bags recur across histories and approaches.

In [ ]:
GLOBAL_KEYS = ["case_study", "global_bag_id", "approach", "history", "history_frames"]
matched_by_metric, matched_exports, dataset_rows, global_rows = {}, [], [], []

for metric in METRICS:
    part = bag_metrics[bag_metrics["metric"] == metric]
    original = part.pivot(index=GLOBAL_KEYS, columns="model", values="original_value").reindex(columns=MODELS)
    oriented = part.pivot(index=GLOBAL_KEYS, columns="model", values="oriented_value").reindex(columns=MODELS)
    original.columns, oriented.columns = ([f"{m}_original" for m in MODELS],
                                          [f"{m}_oriented" for m in MODELS])
    matched = original.join(oriented).reset_index()
    oriented_columns = [f"{m}_oriented" for m in MODELS]
    matched["complete_match"] = matched[oriented_columns].notna().all(axis=1)
    matched.insert(0, "metric", metric)
    matched.insert(1, "direction", DIRECTION[metric])
    assert len(matched) == EXPECTED_BAGS * len(APPROACHES) * len(HISTORIES) == 800
    matched_by_metric[metric] = matched
    matched_exports.append(matched)
    dataset_rows.append({
        "metric": metric, "direction": DIRECTION[metric],
        "possible_configurations": len(matched),
        "complete_configurations": int(matched["complete_match"].sum()),
        "excluded_configurations": int((~matched["complete_match"]).sum()),
        **{f"{m}_valid_values": int(matched[f"{m}_oriented"].notna().sum()) for m in MODELS},
    })

    data = matched[matched["complete_match"]].copy()
    wide = pd.DataFrame({m: data[f"{m}_oriented"] for m in MODELS})
    n, performed = len(wide), len(wide) >= MIN_COMPLETE_BAGS
    statistic, raw_p = safe_friedman(wide) if performed else (np.nan, np.nan)
    global_rows.append({
        "metric": metric, "direction": DIRECTION[metric],
        "possible_configurations": len(matched), "n_complete_configurations": n,
        "n_excluded_configurations": len(matched) - n,
        **{f"{m}_median_original": data[f"{m}_original"].median() for m in MODELS},
        **{f"{m}_median_oriented": data[f"{m}_oriented"].median() for m in MODELS},
        "best_median_model": max(MODELS, key=lambda m: wide[m].median()) if n else "unavailable",
        "friedman_chi2": statistic, "degrees_of_freedom": len(MODELS) - 1,
        "raw_p": raw_p, "kendalls_w": statistic / (n * (len(MODELS) - 1)) if performed else np.nan,
    })

rq2c_matched = pd.concat(matched_exports, ignore_index=True)
rq2c_dataset_summary = pd.DataFrame(dataset_rows)
rq2c_friedman = pd.DataFrame(global_rows)
rq2c_friedman["holm_p"] = holm_adjust(rq2c_friedman["raw_p"])
rq2c_friedman["significant_holm"] = rq2c_friedman["holm_p"] < ALPHA

posthoc_rows = []
for row in rq2c_friedman.itertuples(index=False):
    if row.significant_holm:
        data = matched_by_metric[row.metric].query("complete_match")
        wide = pd.DataFrame({m: data[f"{m}_oriented"] for m in MODELS})
        posthoc_rows.extend(model_pairwise_tests(
            wide, {"metric": row.metric, "direction": row.direction,
                   "friedman_holm_p": row.holm_p}
        ))
rq2c_posthoc = pd.DataFrame(posthoc_rows)

decision_rows = []
for row in rq2c_friedman.itertuples(index=False):
    winner, reason = "No unique winner", "Omnibus not significant"
    if row.significant_holm:
        winner = strict_winner(rq2c_posthoc.query("metric == @row.metric"))
        reason = ("Significantly better than both alternatives" if winner in MODELS
                  else "No model significantly beat both alternatives")
    decision_rows.append({
        "metric": row.metric, "direction": row.direction,
        "n_complete_configurations": row.n_complete_configurations,
        "friedman_chi2": row.friedman_chi2, "friedman_holm_p": row.holm_p,
        "kendalls_w": row.kendalls_w, "unique_winner": winner, "decision_reason": reason,
    })
rq2c_decisions = pd.DataFrame(decision_rows)

summary_rows = []
for metric in METRICS:
    results = rq2c_posthoc[rq2c_posthoc["metric"] == metric]
    for model in MODELS:
        involving = results[(results["model_1"] == model) | (results["model_2"] == model)]
        wins = int(involving["significant_better_model"].eq(model).sum())
        losses = int((involving["significant_holm"] & involving["significant_better_model"].isin(MODELS)
                      & ~involving["significant_better_model"].eq(model)).sum())
        summary_rows.append({"metric": metric, "model": model, "significant_wins": wins,
                             "significant_losses": losses, "tests_involving_model": len(involving),
                             "net_significant_wins": wins - losses})
rq2c_pairwise_summary = pd.DataFrame(summary_rows)

export(rq2c_matched, "rq2c_matched_values", "Matched values used in exploratory RQ2-C.")
export(rq2c_dataset_summary, "rq2c_dataset_summary", "RQ2-C complete-case counts by metric.")
export(rq2c_friedman, "rq2c_global_friedman", "RQ2-C global Friedman model comparisons.")
export(rq2c_posthoc, "rq2c_global_pairwise_wilcoxon",
       "RQ2-C gated global pairwise Wilcoxon-Holm comparisons.")
export(rq2c_decisions, "rq2c_final_metric_decisions", "RQ2-C final model decision by metric.")
export(rq2c_pairwise_summary, "rq2c_model_pairwise_summary",
       "RQ2-C significant pairwise wins and losses by model.")

assert len(rq2c_matched) == 4_800 and len(rq2c_friedman) == 6
assert len(rq2c_posthoc) == 3 * int(rq2c_friedman["significant_holm"].sum())
display(rq2c_dataset_summary)
display(rq2c_friedman)
display(rq2c_decisions)

## Final verification and output manifest

The manifest below lists only RQ2-A, RQ2-B, and RQ2-C result files. All listed tables are saved in both CSV and LaTeX format under Combined/rq2.

In [ ]:
manifest_names = sorted(set(EXPORTED) | {"rq2_output_manifest"})
manifest = pd.DataFrame([{
    "table": name,
    "csv": str(OUT / f"{name}.csv"),
    "latex": str(TEX / f"{name}.tex"),
    "format": "CSV + LaTeX",
} for name in manifest_names])
export(manifest, "rq2_output_manifest", "Manifest of final combined RQ2 result files.")

missing_csv = [name for name in manifest_names if not (OUT / f"{name}.csv").is_file()]
missing_tex = [name for name in manifest_names if not (TEX / f"{name}.tex").is_file()]
assert not missing_csv and not missing_tex, (missing_csv, missing_tex)

assert len(validation) == 96
assert len(bag_metrics) == 14_400
assert len(rq2a) == 384
assert all(table.shape == (16, 12) for table in rq2a_tables.values())
assert len(rq2b_friedman) == 192 and rq2b_winner_matrix.shape == (12, 16)
assert len(rq2b_posthoc) == 3 * int(rq2b_friedman["significant_holm"].sum())
assert len(rq2c_matched) == 4_800 and len(rq2c_friedman) == 6
assert len(rq2c_posthoc) == 3 * int(rq2c_friedman["significant_holm"].sum())

print("COMBINED RQ2 COMPLETE")
print("Validated clean files:", len(validation))
print("Bag-level metric rows:", f"{len(bag_metrics):,}")
print("RQ2-A expected-pattern tests:", len(rq2a))
print("RQ2-B Friedman tests:", len(rq2b_friedman))
print("RQ2-B significant Friedman tests:", int(rq2b_friedman["significant_holm"].sum()))
print("RQ2-B gated Wilcoxon tests:", len(rq2b_posthoc))
print("RQ2-C matched rows:", f"{len(rq2c_matched):,}")
print("RQ2-C Friedman tests:", len(rq2c_friedman))
print("RQ2-C gated Wilcoxon tests:", len(rq2c_posthoc))
print("Output folder:", OUT)
display(manifest)